In [1]:
import jax.numpy as jnp

from gaussed.domains import Euclidean
from gaussed.codomains import Codomain
from gaussed.gp.gp_ops.point_eval import Eval
from gaussed.gp.gp_ops.probe import Probe, ProbeStack, as_stack
from gaussed.gp.kernels.rbf import RBFKernel
from gaussed.gp.kernels.matern import MaternKernel
from gaussed.gp.means import ZeroMeanFun
from gaussed.gp.base import GP, PosteriorGP
from gaussed.model import GPModel


In [2]:
from gaussed.backends.base import ExactBackend

In [6]:
from gaussed.backends.solvers.linear_solver import make_solver
from gaussed.backends.solvers.cholesky import chol_logdet_hook, chol_solve_hook, chol_sqrt_hook, cholesky_factor_from_op

from gaussed.backends.solvers.linear_solver import SolverFns, LinearSolverState

solver_fns = SolverFns(chol_solve_hook(), chol_sqrt_hook(), chol_logdet_hook())

backend = ExactBackend(solverfns=solver_fns)

In [7]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from gaussed.domains import Euclidean
from gaussed.codomains import Codomain
from gaussed.gp.gp_ops.point_eval import Eval
from gaussed.gp.gp_ops.probe import Probe
from gaussed.gp.kernels.rbf import RBFKernel, RBFParams
from gaussed.gp.means import ZeroMeanFun
from gaussed.gp.base import GP
from gaussed.model import GPModel
from gaussed.likelihoods.gaussian import GaussianLikelihood

# --- GP prior ---
domain   = Euclidean((1,))
codomain = Codomain((1,))
mean     = ZeroMeanFun()
rbf_params = RBFParams(lengthscale=jnp.array(0.3), amplitude=jnp.array(1.0))
kernel   = RBFKernel(rbf_params)
backend = ExactBackend(solverfns=solver_fns)

gp = GP(domain, codomain, mean, kernel, backend)

# --- Data ---
key = jax.random.PRNGKey(0)
Xtr = jnp.linspace(-3.0, 3.0, 10).reshape(-1, 1)
y_clean = jnp.sin(Xtr).squeeze()
noise = 0.1
y = y_clean + noise * jax.random.normal(key, shape=y_clean.shape)

# --- Probes ---
Ftr = Probe(ops=(), fnl=Eval(Xtr))
Xt = jnp.linspace(-5.0, 5.0, 300).reshape(-1, 1)
Fte = Probe(ops=(), fnl=Eval(Xt))

# --- Model / condition ---
lik = GaussianLikelihood()
model = GPModel(gp=gp, likelihood=lik)
post = model.condition(Ftr, y)

# --- Prior mean/var (for reference) ---
prior_mean = gp.mean_spec().eval(Xt)[:, 0]
prior_var  = jnp.diag(gp.kernel_spec().k0(Xt, Xt))

# --- Posterior mean/var ---
m_post = post.mean(Fte)
v_post = post.variance(Fte)

# --- Plot ---
xplot = Xt.squeeze()

plt.figure(figsize=(10,5))

# Prior
plt.subplot(1,2,1)
plt.title("Prior GP (RBF)")
plt.fill_between(xplot,
                 prior_mean - 2*jnp.sqrt(prior_var),
                 prior_mean + 2*jnp.sqrt(prior_var),
                 alpha=0.3, label="±2σ")
plt.plot(xplot, prior_mean, "k--", label="mean")
plt.scatter(Xtr.squeeze(), y, s=25, c="r", label="train")
plt.legend()

# Posterior
plt.subplot(1,2,2)
plt.title("Posterior GP")
plt.fill_between(xplot,
                 m_post - 2*jnp.sqrt(v_post),
                 m_post + 2*jnp.sqrt(v_post),
                 alpha=0.3, label="±2σ")
plt.plot(xplot, m_post, label="mean")
plt.scatter(Xtr.squeeze(), y, s=25, c="r", label="train")
plt.legend()

plt.tight_layout()
plt.show()


ValueError: not enough values to unpack (expected 2, got 1)

In [ ]:
        be = backend or self.gp.backend
        ks, ms, ctx = be.make_specs(self)
        rep = be.make_rep(self)

        Fst = as_stack(F)

        # Build training blocks
        mF = rep.mean(Fst, ctx)                 # (n,)
        K_FF = rep.gram(Fst, Fst, ctx)          # (n,n) dense here (Exact)
        A = self.likelihood.add_to_gram(K_FF)   # (n,n)


In [12]:
ks, ms, ctx = backend.make_specs(model)

In [17]:
rep = backend.make_rep(model)
rep

KernelRep(_kernel_spec=KernelSpec(domain=Euclidean(shape=(1,), eps=0.0), k0=<function make_kernel_spec.<locals>.k0 at 0x7f9958341760>, left_shape=(), right_shape=(), integrate_x_of=None, integrate_y_of=None, integrate_xy_of=None, d_dx=None, d_dy=None, d2_xx=None, d2_yy=None, d2_xy=None), _mean_spec=FunSpec(eval=<function make_fun_spec.<locals>.eval at 0x7f9958341800>, integrate=None, partial=None, partial2=None), ctx=OpContext(domain=Euclidean(shape=(1,), eps=0.0), quad=None, quad_x=None, quad_y=None, quad_xy=None))

In [26]:
test = ProbeStack([Probe(ops=(), fnl=Eval(Xt[i])) for i in range(Xt.shape[0])])

In [ ]:
rep.gram()

Array([-5.], dtype=float32)

In [35]:
probe1 = Probe(ops=(), fnl=Eval(jnp.array([[1.0]])))
probe2 = Probe(ops=(), fnl=Eval(jnp.array([[2.0]])))

probe_stack = ProbeStack((probe1, probe2))

rep.gram(probe_stack, probe_stack, ctx)

ValueError: axis 1 is out of bounds for array of dimension 1

In [ ]:
Xt = jnp.linspace(-5.0, 5.0, 10).reshape(-1, 1)
Fte = ProbeStack(Probe(ops=(), fnl=Eval(Xt)))
Fte

Probe(ops=(), fnl=Eval(X=Array([[-5.       ],
       [-3.888889 ],
       [-2.7777777],
       [-1.6666663],
       [-0.5555556],
       [ 0.5555558],
       [ 1.666667 ],
       [ 2.7777781],
       [ 3.888889 ],
       [ 5.       ]], dtype=float32)))

In [27]:
rep.gram(test, test, ctx)

ValueError: Zero-dimensional arrays cannot be concatenated.

In [18]:
rep.mean(Fte, ctx)

Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

In [ ]:
Xt = jnp.linspace(-5.0, 5.0, 10).reshape(-1, 1)
Fte = ProbeStack((Probe(ops=(), fnl=Eval(Xt)),))

rep.gram(Fte, Fte, ctx)

Array([1.3132617, 1.3132617, 1.3132617, 1.3132617, 1.3132617, 1.3132617,
       1.3132617, 1.3132617, 1.3132617, 1.3132617], dtype=float32)

In [16]:
as_stack(Fte)

ProbeStack(probes=(Probe(ops=(), fnl=Eval(X=Array([[-5.       ],
       [-3.888889 ],
       [-2.7777777],
       [-1.6666663],
       [-0.5555556],
       [ 0.5555558],
       [ 1.666667 ],
       [ 2.7777781],
       [ 3.888889 ],
       [ 5.       ]], dtype=float32))),))